# Sound as data: hands-on (optional)
**LLMs From Basics · Lesson A1.1**

Synthesize tones, alias them on purpose, and build a log-mel spectrogram from scratch with NumPy. No audio files or downloads needed.

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; a ✓ means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("ready")

## 1 · Tones and data rates
A pure tone is $x(t) = \sin(2\pi f t + \varphi)$. Sampling at rate $f_s$ means evaluating it at $t = n / f_s$ for $n = 0, 1, 2, \dots$

Implement:
- `tone(f, fs, dur, phase=0.0)`: a float array of `round(fs * dur)` samples.
- `pcm_bitrate(fs, bits, channels=1)`: bits per second of uncompressed PCM.

In [ ]:
def tone(f, fs, dur, phase=0.0):
    # TODO: sample times n / fs, then the sine
    raise NotImplementedError("Exercise 1a")

def pcm_bitrate(fs, bits, channels=1):
    # TODO
    raise NotImplementedError("Exercise 1b")

**Check**

In [ ]:
x = tone(440, 16000, 1.0)
assert x.shape == (16000,)
assert np.isclose(x[0], 0.0) and np.isclose(x[10], np.sin(2 * np.pi * 440 * 10 / 16000))
assert np.isclose(np.max(np.abs(tone(1000, 16000, 0.5))), 1.0, atol=1e-6)
assert pcm_bitrate(16000, 16) == 256_000                 # speech for ASR
assert pcm_bitrate(44100, 16, channels=2) == 1_411_200   # CD
assert pcm_bitrate(24000, 16) == 384_000
print("✓ exercise 1: one minute of 16 kHz/16-bit mono =", pcm_bitrate(16000, 16) * 60 / 8 / 1e6, "MB")

## 2 · Aliasing, predicted and measured
A tone at $f$ sampled at $f_s$ shows up at $f_{\text{alias}} = |f - k f_s|$ with $k$ the whole number nearest $f/f_s$.

1. Implement `alias_freq(f, fs)`.
2. Implement `peak_freq(x, fs)`: the frequency (in Hz) of the largest-magnitude bin of `np.fft.rfft(x)`. Use `np.fft.rfftfreq(len(x), 1 / fs)` for the bin frequencies.

The check samples real tones too slowly and confirms the FFT sees the predicted alias.

In [ ]:
def alias_freq(f, fs):
    # TODO
    raise NotImplementedError("Exercise 2a")

def peak_freq(x, fs):
    # TODO
    raise NotImplementedError("Exercise 2b")

**Check**

In [ ]:
assert alias_freq(7000, 10000) == 3000      # the warm-up question
assert alias_freq(5000, 8000) == 3000
assert alias_freq(3000, 16000) == 3000      # below Nyquist: unchanged
assert alias_freq(9000, 8000) == 1000
for f, fs in [(7000, 10000), (5000, 8000), (3000, 16000), (11000, 16000)]:
    measured = peak_freq(tone(f, fs, 1.0, phase=0.3), fs)
    assert abs(measured - alias_freq(f, fs)) <= 1, (f, fs, measured)
print("✓ exercise 2: an 11 kHz tone sampled at 16 kHz looks like", alias_freq(11000, 16000), "Hz")

## 3 · A short-time Fourier transform by hand
Speech models look at audio in short overlapping windows. Implement `stft_mag(x, n_fft=400, hop=160)`:
1. Cut `x` into frames of length `n_fft`, starting every `hop` samples (no padding: the last frame must fit entirely). That's `1 + (len(x) - n_fft) // hop` frames.
2. Multiply each frame by a Hann window, `np.hanning(n_fft)`.
3. Take `np.abs(np.fft.rfft(frame))`.

Return an array of shape `(n_frames, n_fft // 2 + 1)`. At 16 kHz, `n_fft = 400` is a 25 ms window and `hop = 160` is 10 ms, exactly Whisper's settings.

In [ ]:
def stft_mag(x, n_fft=400, hop=160):
    # TODO
    raise NotImplementedError("Exercise 3")

**Check**

In [ ]:
fs = 16000
S = stft_mag(tone(1000, fs, 1.0))
assert S.shape == (98, 201), S.shape                 # 1 + (16000 - 400) // 160 = 98 frames
bin_hz = fs / 400                                     # 40 Hz per bin
assert np.all(np.argmax(S, axis=1) == round(1000 / bin_hz))   # every frame peaks at bin 25
x = np.random.default_rng(0).normal(size=1000)
ref = np.abs(np.fft.rfft(x[160:560] * np.hanning(400)))
assert np.allclose(stft_mag(x)[1], ref)
print("✓ exercise 3: 1 s of audio ->", S.shape[0], "frames of", S.shape[1], "frequency bins")

## 4 · A mel filterbank
The mel scale squeezes high frequencies together, roughly like hearing does. We use the common "HTK" formula:
$$m = 2595 \log_{10}\!\left(1 + \frac{f}{700}\right), \qquad f = 700\left(10^{m/2595} - 1\right)$$
A filterbank has `n_mels` triangular filters. Their edges are `n_mels + 2` points spaced **evenly in mel** from `fmin` to `fmax`. Filter $i$ (for $i = 1 \dots$ `n_mels`) rises linearly from 0 at edge $i-1$ to 1 at edge $i$, then falls back to 0 at edge $i+1$. Evaluate each filter at the STFT bin frequencies `np.fft.rfftfreq(n_fft, 1 / fs)`.

Implement `hz_to_mel`, `mel_to_hz` and `mel_filterbank(n_mels, n_fft, fs, fmin=0, fmax=None)` returning shape `(n_mels, n_fft // 2 + 1)`. (`fmax` defaults to `fs / 2`.)

In [ ]:
def hz_to_mel(f):
    # TODO
    raise NotImplementedError("Exercise 4a")

def mel_to_hz(m):
    # TODO
    raise NotImplementedError("Exercise 4b")

def mel_filterbank(n_mels, n_fft, fs, fmin=0.0, fmax=None):
    fmax = fs / 2 if fmax is None else fmax
    freqs = np.fft.rfftfreq(n_fft, 1 / fs)
    # TODO: edges evenly spaced in mel, converted back to Hz; then one triangle per filter
    raise NotImplementedError("Exercise 4c")

**Check**

In [ ]:
assert np.isclose(hz_to_mel(1000), 1000, atol=0.5)        # the scale is anchored near 1 kHz
assert np.isclose(mel_to_hz(hz_to_mel(3456.0)), 3456.0)
fb = mel_filterbank(40, 512, 16000)
assert fb.shape == (40, 257)
assert fb.min() >= 0 and fb.max() <= 1 + 1e-9
assert np.all(fb.sum(axis=1) > 0)                          # no empty filters
peaks = np.fft.rfftfreq(512, 1 / 16000)[np.argmax(fb, axis=1)]
assert np.all(np.diff(peaks) >= 0)                          # centers move up in frequency
widths = (fb > 0).sum(axis=1)
assert widths[-1] > 4 * widths[0]                           # high filters are much wider
print("✓ exercise 4: filter 1 spans", widths[0], "bins, filter 40 spans", widths[-1], "bins")

## 5 · The log-mel spectrogram
Put it together: `log_mel(x, fs=16000, n_fft=400, hop=160, n_mels=80)`:
1. power spectrum: `stft_mag(x, n_fft, hop) ** 2`, shape `(frames, bins)`;
2. apply the filterbank: `fb @ power.T`, shape `(n_mels, frames)`;
3. take `np.log10` of it plus a small floor (`1e-10`) so silence doesn't give `-inf`.

This is (up to scaling and clipping details) what Whisper feeds its encoder.

In [ ]:
def log_mel(x, fs=16000, n_fft=400, hop=160, n_mels=80):
    # TODO
    raise NotImplementedError("Exercise 5")

**Check**

In [ ]:
fs = 16000
L = log_mel(tone(1000, fs, 1.0))
assert L.shape == (80, 98)
assert np.isfinite(L).all()
fb = mel_filterbank(80, 400, fs)
centers = np.fft.rfftfreq(400, 1 / fs)[np.argmax(fb, axis=1)]
strongest = np.argmax(L.mean(axis=1))
assert abs(centers[strongest] - 1000) <= 80, centers[strongest]
print("✓ exercise 5: a 1 kHz tone lights up mel band", strongest, "of 80 (center ≈", int(centers[strongest]), "Hz)")

## Experiment

A **chirp** sweeps from 200 Hz to 7 kHz. Look at it as a linear spectrogram and as a log-mel spectrogram: the mel version gives low frequencies more rows.

In [ ]:
fs = 16000
t = np.arange(2 * fs) / fs
chirp = np.sin(2 * np.pi * (200 * t + (7000 - 200) / (2 * 2) * t ** 2))   # 200 Hz -> 7 kHz over 2 s
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].imshow(np.log10(stft_mag(chirp).T ** 2 + 1e-10), origin="lower", aspect="auto", extent=[0, 2, 0, fs / 2])
ax[0].set(title="linear-frequency spectrogram", xlabel="time (s)", ylabel="Hz")
ax[1].imshow(log_mel(chirp), origin="lower", aspect="auto", extent=[0, 2, 0, 80])
ax[1].set(title="log-mel spectrogram (80 bands)", xlabel="time (s)", ylabel="mel band")
plt.tight_layout(); plt.show()

Now **downsample without filtering**: keep every 4th sample, so 16 kHz becomes 4 kHz (Nyquist 2 kHz). Above 2 kHz the chirp folds back and appears to sweep *down*. Then try a crude low-pass first (a moving average) and compare. Real resamplers use much better filters.

In [ ]:
bad = chirp[::4]
smooth = np.convolve(chirp, np.ones(8) / 8, mode="same")[::4]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for a, sig, title in [(ax[0], bad, "no filter: aliasing"), (ax[1], smooth, "moving-average filter first")]:
    a.imshow(np.log10(stft_mag(sig, n_fft=100, hop=40).T ** 2 + 1e-10), origin="lower", aspect="auto", extent=[0, 2, 0, 2000])
    a.set(title=title, xlabel="time (s)", ylabel="Hz (fs = 4 kHz)")
plt.tight_layout(); plt.show()

**Try:** change `n_mels` to 128 (Whisper large-v3) or the hop to 20 ms. How many frames does 30 s give now? How would that change a Transformer's cost?

## Recap
- PCM = samples at $f_s$, each with $b$ bits: data rate $f_s \times b \times$ channels.
- Tones above $f_s/2$ alias to $|f - k f_s|$; filter before you sample or resample.
- STFT (25 ms Hann windows, 10 ms hop) → mel filterbank → log gives ~100 frames per second, the standard ASR input.

Back to the course to finish the lesson and add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def tone(f, fs, dur, phase=0.0):
    n = np.arange(int(round(fs * dur)))
    return np.sin(2 * np.pi * f * n / fs + phase)

def pcm_bitrate(fs, bits, channels=1):
    return fs * bits * channels

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def alias_freq(f, fs):
    k = round(f / fs)
    return abs(f - k * fs)

def peak_freq(x, fs):
    spec = np.abs(np.fft.rfft(x))
    freqs = np.fft.rfftfreq(len(x), 1 / fs)
    return freqs[np.argmax(spec)]

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def stft_mag(x, n_fft=400, hop=160):
    n_frames = 1 + (len(x) - n_fft) // hop
    win = np.hanning(n_fft)
    frames = np.stack([x[i * hop: i * hop + n_fft] * win for i in range(n_frames)])
    return np.abs(np.fft.rfft(frames, axis=1))

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def hz_to_mel(f):
    return 2595 * np.log10(1 + np.asarray(f, dtype=float) / 700)

def mel_to_hz(m):
    return 700 * (10 ** (np.asarray(m, dtype=float) / 2595) - 1)

def mel_filterbank(n_mels, n_fft, fs, fmin=0.0, fmax=None):
    fmax = fs / 2 if fmax is None else fmax
    freqs = np.fft.rfftfreq(n_fft, 1 / fs)
    edges = mel_to_hz(np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_mels + 2))
    fb = np.zeros((n_mels, len(freqs)))
    for i in range(1, n_mels + 1):
        lo, c, hi = edges[i - 1], edges[i], edges[i + 1]
        up = (freqs - lo) / (c - lo)
        down = (hi - freqs) / (hi - c)
        fb[i - 1] = np.maximum(0, np.minimum(up, down))
    return fb

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def log_mel(x, fs=16000, n_fft=400, hop=160, n_mels=80):
    power = stft_mag(x, n_fft, hop) ** 2
    fb = mel_filterbank(n_mels, n_fft, fs)
    return np.log10(fb @ power.T + 1e-10)